<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z320_AutoGluon_Racha_Galactus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoGluon + Test de Racha + Galactus

## ¿Para qué sirve este experimento?

Este notebook valida si el Test de Racha está detectando estructura temporal real o simplemente clasificando ruido como señal.

```
z319  AutoGluon + Racha (sin shuffle)   →  score A   ← referencia
z320  AutoGluon + Racha + Galactus      →  score B   ← este notebook
```

| Resultado | Interpretación |
|---|---|
| score B ≈ score A | El Test de Racha se equivocó: las series "con estructura" eran ruido igual |
| score B >> score A | El Test de Racha acertó: había señal real que AutoGluon aprovechaba |

El shuffle Galactus se aplica **solo** a las series que el Test de Racha clasificó como estructuradas. Las series ruidosas siguen yendo directo al promedio 12m, igual que en z319.

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle
!uv pip install autogluon[all]

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime
from scipy.stats import runstest_1samp
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

import warnings
warnings.filterwarnings('ignore')

Usar la misma semilla y alpha que en `z319` para que la comparacion sea apareada.

In [ ]:
PARAM = {
  'experimento': 'AutoGluon_Racha_Galactus-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  'semilla_primigenia': 102191,
  'alpha_runs': 0.05
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Preparacion de datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")

tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])
print(f"Productos x meses: {tb_ventas.height} filas")

tb_ventas = tb_ventas.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

# 3  Test de Racha por producto

In [ ]:
productos = tb_apredecir["product_id"].to_list()
runs_resultados = []

for producto in productos:
    serie = (
        tb_ventas
        .filter(pl.col("product_id") == producto)
        .sort("periodo")["tn"]
        .to_numpy().astype(float)
    )

    try:
        _, pvalue = runstest_1samp(serie, cutoff='median')
        tiene_estructura = bool(pvalue < PARAM['alpha_runs'])
    except Exception:
        pvalue = np.nan
        tiene_estructura = False

    promedio_12m = float(serie[-12:].mean()) if len(serie) >= 12 else float(serie.mean())

    runs_resultados.append({
        'product_id': producto,
        'pvalue_runs': round(float(pvalue), 4) if not np.isnan(pvalue) else None,
        'tiene_estructura': tiene_estructura,
        'promedio_12m': promedio_12m
    })

tb_runs = pl.DataFrame(runs_resultados)

n_total      = len(productos)
n_estructura = tb_runs["tiene_estructura"].sum()
n_ruido      = n_total - n_estructura
print(f"Total          : {n_total}")
print(f"Con estructura : {n_estructura}  ({100*n_estructura/n_total:.1f}%)  → Galactus + AutoGluon")
print(f"Sin estructura : {n_ruido}  ({100*n_ruido/n_total:.1f}%)  → promedio 12m directo")

# 4  Visualizacion del Test de Racha

## 4.1 Distribucion de p-values

Un histograma de los p-values nos dice cuán seguros estamos de la clasificación. P-values muy bajos (< 0.05) indican estructura clara. P-values altos indican series que son estadísticamente indistinguibles del ruido.

In [ ]:
pvalues = tb_runs.drop_nulls("pvalue_runs")["pvalue_runs"].to_numpy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pvalues, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(PARAM['alpha_runs'], color='red', linestyle='--', linewidth=1.5,
           label=f"alpha = {PARAM['alpha_runs']}  |  izq: {n_estructura} con estructura  |  der: {n_ruido} ruido")
ax.set_xlabel('p-value Test de Racha')
ax.set_ylabel('cantidad de productos')
ax.set_title('Distribucion de p-values — Test de Racha (780 productos)')
ax.legend()
plt.tight_layout()
plt.show()

## 4.2 Series que NO pasaron el Test de Racha (ruido)

Mostramos 6 series con p-value alto — las que el test clasificó como ruido y van directo al promedio 12m sin pasar por AutoGluon.

In [ ]:
# los 6 productos con p-value mas alto (los mas "ruidosos")
productos_ruido = (
    tb_runs
    .filter(pl.col("tiene_estructura") == False)
    .sort("pvalue_runs", descending=True)
    .head(6)["product_id"]
    .to_list()
)

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
axes = axes.flatten()

for i, pid in enumerate(productos_ruido):
    serie = (
        tb_ventas.filter(pl.col("product_id") == pid)
        .sort("periodo")
    )
    pval = tb_runs.filter(pl.col("product_id") == pid)["pvalue_runs"][0]

    axes[i].plot(serie["timestamp"].to_list(), serie["tn"].to_list(),
                 color='tomato', linewidth=1.5, marker='o', markersize=3)
    axes[i].set_title(f"product_id {pid}\np-value = {pval:.3f}  →  RUIDO", fontsize=9)
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)
    axes[i].set_ylabel('tn', fontsize=8)

fig.suptitle('Series que NO pasaron el Test de Racha (ruido — van a promedio 12m)', fontsize=11)
plt.tight_layout()
plt.show()

## 4.3 Series que SÍ pasaron el Test de Racha (con estructura)

Mostramos 6 series con p-value bajo — las que el test clasificó como estructuradas y van a AutoGluon (con shuffle Galactus en este notebook).

In [ ]:
# los 6 productos con p-value mas bajo (los mas "estructurados")
productos_estructura = (
    tb_runs
    .filter(pl.col("tiene_estructura") == True)
    .sort("pvalue_runs")
    .head(6)["product_id"]
    .to_list()
)

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
axes = axes.flatten()

for i, pid in enumerate(productos_estructura):
    serie = (
        tb_ventas.filter(pl.col("product_id") == pid)
        .sort("periodo")
    )
    pval = tb_runs.filter(pl.col("product_id") == pid)["pvalue_runs"][0]

    axes[i].plot(serie["timestamp"].to_list(), serie["tn"].to_list(),
                 color='steelblue', linewidth=1.5, marker='o', markersize=3)
    axes[i].set_title(f"product_id {pid}\np-value = {pval:.4f}  →  ESTRUCTURA", fontsize=9)
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)
    axes[i].set_ylabel('tn', fontsize=8)

fig.suptitle('Series que SÍ pasaron el Test de Racha (estructura — van a AutoGluon + Galactus)', fontsize=11)
plt.tight_layout()
plt.show()

# 5  Shuffle Galactus — solo series CON estructura

Destruimos la estructura temporal de las series que el Test de Racha identificó como no aleatorias. Si AutoGluon pierde mucho con el shuffle, el test estaba detectando señal real.

In [ ]:
productos_con_estructura = (
    tb_runs.filter(pl.col("tiene_estructura") == True)["product_id"].to_list()
)

tb_ventas_ag = tb_ventas.filter(
    pl.col("product_id").is_in(productos_con_estructura)
).sort(["product_id", "periodo"])

np.random.seed(PARAM['semilla_primigenia'])

piezas = []
for pid in tb_ventas_ag["product_id"].unique(maintain_order=True).to_list():
    sub = tb_ventas_ag.filter(pl.col("product_id") == pid)
    tn_vals = sub["tn"].to_numpy().copy()
    np.random.shuffle(tn_vals)
    piezas.append(sub.with_columns(pl.Series("tn", tn_vals)))

tb_ventas_ag = pl.concat(piezas).sort(["product_id", "periodo"])

print(f"Galactus aplicado sobre {len(productos_con_estructura)} series")

## 5.1 Antes vs despues del shuffle — mismas 6 series estructuradas

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(14, 18))

for i, pid in enumerate(productos_estructura):
    # serie original
    serie_orig = (
        tb_ventas.filter(pl.col("product_id") == pid).sort("periodo")
    )
    # serie shuffleada
    serie_shuf = (
        tb_ventas_ag.filter(pl.col("product_id") == pid).sort("periodo")
    )

    ts = serie_orig["timestamp"].to_list()

    axes[i, 0].plot(ts, serie_orig["tn"].to_list(), color='steelblue', linewidth=1.5, marker='o', markersize=3)
    axes[i, 0].set_title(f"{pid} — ORIGINAL", fontsize=9)
    axes[i, 0].tick_params(axis='x', rotation=45, labelsize=7)

    axes[i, 1].plot(ts, serie_shuf["tn"].to_list(), color='orange', linewidth=1.5, marker='o', markersize=3)
    axes[i, 1].set_title(f"{pid} — GALACTUS (shuffleada)", fontsize=9)
    axes[i, 1].tick_params(axis='x', rotation=45, labelsize=7)

fig.suptitle('Antes vs despues del shuffle Galactus — 6 series con estructura', fontsize=11)
plt.tight_layout()
plt.show()

# 6  Entrenamiento AutoGluon sobre series shuffleadas

In [ ]:
ts_data = TimeSeriesDataFrame.from_data_frame(
  tb_ventas_ag.to_pandas(),
  timestamp_column='timestamp',
  id_column='product_id'
)

In [ ]:
global_eval_metric = 'RMSE'

modelo = TimeSeriesPredictor(
  prediction_length= 2,
  target= 'tn',
  freq= 'MS',
  eval_metric= global_eval_metric
)

modelo.fit(ts_data,
  num_val_windows= 2,
  time_limit= 3600,
  presets= "best_quality",
  random_seed= PARAM['semilla_primigenia']
)

# 7  Prediccion + ensamble con promedio 12m

In [ ]:
tb_forecast = modelo.predict(ts_data, random_seed=PARAM['semilla_primigenia'])
tb_forecast = pl.from_pandas(tb_forecast.reset_index())

tb_ag_final = (
    tb_forecast
    .filter(pl.col("timestamp") == datetime(2020, 2, 1))
    .select(["item_id", "mean"])
    .rename({"item_id": "product_id", "mean": "tn"})
)

tb_ruido_final = (
    tb_runs
    .filter(pl.col("tiene_estructura") == False)
    .select(["product_id", "promedio_12m"])
    .rename({"promedio_12m": "tn"})
)

print(f"AutoGluon (shuffleados): {tb_ag_final.height}")
print(f"Promedio 12m (ruido)   : {tb_ruido_final.height}")

tb_final = pl.concat([tb_ag_final, tb_ruido_final]).sort("product_id")
tb_final = tb_final.with_columns(
    pl.when(pl.col("tn") < 0).then(0.0).otherwise(pl.col("tn")).alias("tn")
)

display(tb_final)
print(f"Total: {tb_final.height}  |  Nulls: {tb_final['tn'].is_null().sum()}")

# 8  Submit a Kaggle

In [ ]:
archivo = "AutoGluon_Racha_Galactus_" + global_eval_metric + ".csv"
mensaje = "AutoGluon Racha+Galactus alpha=" + str(PARAM['alpha_runs']) + " " + global_eval_metric

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)

# 9  Tabla de comparacion final

| Notebook | Descripcion | Score Kaggle |
|---|---|---|
| z316 | AutoGluon base (todas las series) | |
| z319 | AutoGluon + Racha sin shuffle | |
| z320 | AutoGluon + Racha + Galactus | |

**Lectura**:
- `z319 < z316` → filtrar las series ruidosas mejoró el score
- `z320 >> z319` → el Test de Racha detectaba estructura real que AutoGluon aprovechaba ✓
- `z320 ≈ z319` → AutoGluon en esas series estaba cerca de un promedio de todos modos